In [1]:
import jax
jax.config.update("jax_enable_x64", True)  # Enable 64-bit precision
jax.config.update("jax_platforms", "cpu")
import jax.numpy as jnp
import numpyro
numpyro.set_host_device_count(4)
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
import numpy as np
import arviz as az
import matplotlib.pyplot as plt
import orthax
import functools
import pandas as pd

from typing import Dict, Tuple, NamedTuple


In [2]:
def scatter_vector(theta, lambda_0=633e-9, n=1.33, radians=False):
    """
    theta - scatter angle
    lambda_0 - lsder wavelength in meters
    n - refractive index of water
    """
    if not radians:
        theta = jnp.radians(theta)
    return (4 * jnp.pi * n/lambda_0) * jnp.sin(theta / 2)

def diffusion_coef(r, k_B=1.38e-23, T=298.15, eta=0.00089):
    """
    r - particle radius
    k_b - Boltzmann constant (J/K)
    T - Temperature (K)
    eta - Viscosity of water at room temerature (Pa*s)
    """
    return k_B * T / (6 * jnp.pi * eta * r)

def normal_to_lognormal_params(mu, sigma):
    sigma_ln_sq = jnp.log(1 + (sigma / mu)**2)
    mu_ln = jnp.log(mu) - (sigma_ln_sq / 2)
    sigma_ln = jnp.sqrt(sigma_ln_sq)
    return (mu_ln, sigma_ln)

def prep_data(datafile):
    df = pd.read_csv(datafile, delimiter='\t', header=None)
    df = df.iloc[1:] # remove strange first point
    d = jnp.array(df.to_numpy())

    t = d[:, 0]
    t *=  1e-3 # convert timestamps from ms to s

    # observation data is g2(t) - 1
    g2_minus1_obs = d[:, 1:].T

    div = jax.vmap(lambda x: x/x[0])
    g1_squared = div(g2_minus1_obs) # normalization by first term handles removing the beta term (roughly)

    # g1 = jnp.sqrt(jnp.maximum(g1_squared, 0)) # this line converts to g1

    g1 = jnp.where(
        jnp.greater_equal(g1_squared, 0),
        jnp.sqrt(g1_squared),
        -jnp.sqrt(-g1_squared)
    )

    # g1 = jnp.sign(g1_squared) * jnp.sqrt(jnp.abs(g1_squared))

    theta = jnp.arange(30., 151, 5) # angles known in advance - in degrees
    q = scatter_vector(theta)
    return q, t, g1

def prep_data_g2(datafile):
    df = pd.read_csv(datafile, delimiter='\t', header=None)
    df = df.iloc[1:] # remove strange first point
    d = jnp.array(df.to_numpy())

    t = d[:, 0]
    t *=  1e-3 # convert timestamps from ms to s

    # observation data is g2(t) - 1
    g2_minus1_obs = d[:, 1:].T

    div = jax.vmap(lambda x: x/x[0])
    g1_squared = div(g2_minus1_obs) # normalization by first term handles removing the beta term (roughly)

    # # g1 = jnp.sqrt(jnp.maximum(g1_squared, 0)) # this line converts to g1
    # g1 = jnp.where(
    #     jnp.greater_equal(g1_squared, 0),
    #     jnp.sqrt(g1_squared),
    #     -jnp.sqrt(-g1_squared)
    # )

    theta = jnp.arange(30., 151, 5) # angles known in advance - in degrees
    q = scatter_vector(theta)
    # return q, t, g2_minus1_obs
    return q, t, g1_squared



In [3]:
### models
SCALING_CONST = 2.45e-12


######### Normal model ##########

def get_g1(t, nk, a, c):
    b = a**2/2
    d = jnp.sqrt(2)
    s_pi = jnp.sqrt(jnp.pi)
    x = (-a + c*t)/d
    y = jnp.where(
        jnp.greater_equal(x, 5),
        1/(x*s_pi),
        jax.scipy.special.erfc(x)*jnp.exp(x**2)
    )
    e = (nk/2)*jnp.exp(-b)
    return e*y

# vectorize along time dimension
all_g1 = jax.vmap(
    get_g1,
    in_axes=(0, None, None, None)
)

def source3_1(t, q, amp, mu, sig):
    ################
    const = SCALING_CONST
    # const = 1.0
    ################
    nk = amp
    sig = sig*const
    mu = mu*const
    a = mu/sig
    c = q**2*sig

    g = all_g1(t, nk, a, c)
    return g

by_Xs1 = jax.vmap(
    source3_1,
    in_axes=(None, None, 0, 0, 0)
)

source_matrix1 = jax.vmap(
    by_Xs1,
    in_axes=(None, 0, None, None, None)
)

def g1_matrix(q, t, amp, mu, sig):
    full = source_matrix1(t, q, amp, mu, sig)
    full = jnp.sum(full, axis=1)
    return full

@jax.jit
def g2_minus1_matrix(q, t, amp, mu, sig, beta):
    g1 = g1_matrix(q, t, amp, mu, sig)
    g2_minus1 = beta * g1**2
    return g2_minus1



In [ ]:
class GMM_State(NamedTuple):
    """State for variable-dimension GMM"""
    k: int  # number of components
    amp: jnp.ndarray  # amplitudes (k-dimensional)
    mu: jnp.ndarray  # means (k-dimensional)
    sig: jnp.ndarray  # sigmas (k-dimensional)
    beta: float
    noise_std: float
    log_prob: float  # log posterior probability

def pad_parameters(params: Dict, max_k: int) -> Dict:
    """Pad parameters to max_k dimension for JAX compatibility"""
    k = params['k']
    padded = params.copy()
    
    # Pad arrays with zeros (or inactive values)
    padded['amp'] = jnp.pad(params['amp'], (0, max_k - k), constant_values=0)
    padded['mu'] = jnp.pad(params['mu'], (0, max_k - k), constant_values=1e-10)
    padded['sig'] = jnp.pad(params['sig'], (0, max_k - k), constant_values=1e-10)
    
    return padded

def compute_log_posterior(state: GMM_State, q, t, observed_g2, priors):
    """Compute log posterior for current state"""
    # Only use first k components
    amp_active = state.amp[:state.k]
    mu_active = state.mu[:state.k]
    sig_active = state.sig[:state.k]
    
    # Ensure amplitudes sum to 1
    amp_active = amp_active / jnp.sum(amp_active)
    
    # Compute model prediction using your existing function
    # You'll need to modify g2_minus1_quadrature_scaled to handle variable k
    g2_predicted = g2_minus1_matrix(
        q, t, amp_active, mu_active, sig_active, state.beta
    )
    
    # Log likelihood
    log_lik = jnp.sum(dist.Normal(g2_predicted.flatten(), state.noise_std).log_prob(observed_g2.flatten()))
    
    # Log priors
    log_prior = 0.0
    
    # Prior on k (Poisson or geometric)
    log_prior += dist.Poisson(priors['k_mean']).log_prob(state.k)
    
    # Prior on amplitudes (Dirichlet)
    if state.k > 1:
        log_prior += dist.Dirichlet(jnp.ones(state.k)).log_prob(amp_active)
    
    # Priors on mu and sig (using your lognormal approach)
    for i in range(state.k):
        log_prior += dist.LogNormal(priors['mu_loc'], priors['mu_scale']).log_prob(mu_active[i])
        log_prior += dist.LogNormal(priors['sig_loc'], priors['sig_scale']).log_prob(sig_active[i])
    
    # Prior on beta
    log_prior += dist.Beta(7, 3).log_prob(state.beta)
    
    # Prior on noise
    log_prior += dist.HalfNormal(priors['noise_scale']).log_prob(state.noise_std)
    
    return log_lik + log_prior

def birth_move(state: GMM_State, rng_key, priors, max_k: int) -> Tuple[GMM_State, float]:
    """Birth move: add a component"""
    if state.k >= max_k:
        return state, -jnp.inf  # Reject if at max
    
    keys = jax.random.split(rng_key, 4)
    
    # Sample new component parameters
    new_mu = jax.random.lognormal(keys[0], shape=()) * priors['mu_scale'] + priors['mu_loc']
    new_sig = jax.random.lognormal(keys[1], shape=()) * priors['sig_scale'] + priors['sig_loc']
    
    # Split amplitude from existing component
    split_idx = jax.random.choice(keys[2], state.k)
    new_amp = state.amp.at[split_idx].multiply(0.5)
    new_amp = new_amp.at[state.k].set(state.amp[split_idx] * 0.5)
    
    # Create new state
    new_mu_array = state.mu.at[state.k].set(new_mu)
    new_sig_array = state.sig.at[state.k].set(new_sig)
    
    new_state = GMM_State(
        k=state.k + 1,
        amp=new_amp,
        mu=new_mu_array,
        sig=new_sig_array,
        beta=state.beta,
        noise_std=state.noise_std,
        log_prob=0.0  # Will be computed
    )
    
    # Compute acceptance ratio (simplified - you need proper Jacobian)
    log_acceptance = jnp.log(state.k + 1) - jnp.log(priors['k_mean'])
    
    return new_state, log_acceptance

def death_move(state: GMM_State, rng_key, priors) -> Tuple[GMM_State, float]:
    """Death move: remove a component"""
    if state.k <= 1:
        return state, -jnp.inf  # Reject if only one component
    
    # Choose component to remove
    remove_idx = jax.random.choice(rng_key, state.k)
    
    # Redistribute amplitude to remaining components
    removed_amp = state.amp[remove_idx]
    new_amp = state.amp.copy()
    new_amp = new_amp.at[remove_idx].set(0)
    
    # Redistribute proportionally
    if state.k > 1:
        remaining_total = jnp.sum(new_amp)
        if remaining_total > 0:
            new_amp = new_amp * (1.0 / remaining_total)
    
    # Shift components down if needed (simplified)
    # In practice, you'd want to maintain ordering
    
    new_state = GMM_State(
        k=state.k - 1,
        amp=new_amp,
        mu=state.mu,  # Keep same array, just use fewer elements
        sig=state.sig,
        beta=state.beta,
        noise_std=state.noise_std,
        log_prob=0.0
    )
    
    # Compute acceptance ratio
    log_acceptance = jnp.log(priors['k_mean']) - jnp.log(state.k)
    
    return new_state, log_acceptance

def rjmcmc_step(state: GMM_State, rng_key, q, t, observed_g2, priors, max_k: int):
    """Single RJMCMC step"""
    keys = jax.random.split(rng_key, 5)
    
    # Decide move type
    move_prob = jax.random.uniform(keys[0])
    
    # 30% chance of dimension change, 70% within-model moves
    if move_prob < 0.15:  # Birth
        proposal, log_acceptance = birth_move(state, keys[1], priors, max_k)
    elif move_prob < 0.30:  # Death
        proposal, log_acceptance = death_move(state, keys[2], priors)
    else:  # Within-model update
        proposal = within_model_update(state, keys[3], priors)
        log_acceptance = 0.0
    
    # Compute log posteriors
    current_log_post = compute_log_posterior(state, q, t, observed_g2, priors)
    proposal_log_post = compute_log_posterior(proposal, q, t, observed_g2, priors)
    
    # Metropolis-Hastings accept/reject
    log_alpha = proposal_log_post - current_log_post + log_acceptance
    accept = jnp.log(jax.random.uniform(keys[4])) < log_alpha
    
    return jax.lax.cond(
        accept,
        lambda: proposal._replace(log_prob=proposal_log_post),
        lambda: state
    )

def within_model_update(state: GMM_State, rng_key, priors):
    """Update parameters within current model dimension"""
    keys = jax.random.split(rng_key, 5)
    
    # Simple random walk updates (you can make this more sophisticated)
    # Update only active components
    k = state.k
    
    # Update amplitudes (maintaining sum to 1)
    amp_noise = jax.random.normal(keys[0], shape=(k,)) * 0.01
    new_amp = state.amp.at[:k].add(amp_noise)
    new_amp = new_amp.at[:k].set(jnp.abs(new_amp[:k]))  # Ensure positive
    new_amp = new_amp.at[:k].set(new_amp[:k] / jnp.sum(new_amp[:k]))  # Normalize
    
    # Update mu
    mu_noise = jax.random.normal(keys[1], shape=(k,)) * 0.01
    new_mu = state.mu.at[:k].add(mu_noise)
    new_mu = jnp.abs(new_mu)  # Ensure positive
    
    # Update sig
    sig_noise = jax.random.normal(keys[2], shape=(k,)) * 0.01
    new_sig = state.sig.at[:k].add(sig_noise)
    new_sig = jnp.abs(new_sig)  # Ensure positive
    
    # Update beta
    beta_noise = jax.random.normal(keys[3]) * 0.01
    new_beta = jnp.clip(state.beta + beta_noise, 0.01, 0.99)
    
    # Update noise_std
    noise_noise = jax.random.normal(keys[4]) * 0.001
    new_noise_std = jnp.abs(state.noise_std + noise_noise)
    
    return GMM_State(
        k=state.k,
        amp=new_amp,
        mu=new_mu,
        sig=new_sig,
        beta=new_beta,
        noise_std=new_noise_std,
        log_prob=0.0
    )

def run_rjmcmc(q, t, observed_g2, num_iterations=10000, max_k=5, rng_key=jax.random.PRNGKey(42)):
    """Main RJMCMC loop"""
    
    # Set priors
    priors = {
        'k_mean': 2.0,  # Expected number of components
        'mu_loc': 0.5,
        'mu_scale': 0.3,
        'sig_loc': 0.2,
        'sig_scale': 0.1,
        'noise_scale': 0.01
    }
    
    # Initialize with k=2 components
    initial_state = GMM_State(
        k=2,
        amp=jnp.array([0.5, 0.5, 0, 0, 0])[:max_k],  # Padded
        mu=jnp.array([0.7, 0.1, 0, 0, 0])[:max_k],
        sig=jnp.array([0.14, 0.03, 0, 0, 0])[:max_k],
        beta=0.9,
        noise_std=0.001,
        log_prob=0.0
    )
    
    # Storage for samples
    samples = []
    state = initial_state
    
    # Main MCMC loop
    for i in range(num_iterations):
        rng_key, step_key = jax.random.split(rng_key)
        state = rjmcmc_step(state, step_key, q, t, observed_g2, priors, max_k)
        
        if i % 100 == 0:
            print(f"Iteration {i}, k={state.k}, log_prob={state.log_prob:.2f}")
        
        if i > num_iterations // 2:  # Collect samples after burn-in
            samples.append(state)
    
    return samples

In [5]:
# Load your data
rng_key=jax.random.PRNGKey(42)
rng_key, model_key, data_key = jax.random.split(rng_key, 3)

q_vals, t_vals, _ = prep_data_g2("~/repos/DLS/Experimental_data_083122/mix_1.csv")

true_params = {
    'beta': 1.0,
    'amp': jnp.array([0.5, 0.5]),
    'mu': jnp.array([0.9, 0.3]),
    'sig': jnp.array([0.25, 0.08]),
    'loss': 0.01,
    'noise_std': 0.001
}

true_g2 = g2_minus1_matrix(
    q_vals, t_vals,
    true_params['amp'], true_params['mu'], true_params['sig'], true_params['beta']
)
observed_g2 = true_g2 + true_params['noise_std'] * jax.random.normal(data_key, shape=true_g2.shape)

# Run RJMCMC
samples = run_rjmcmc(q_vals, t_vals, observed_g2, num_iterations=1100, max_k=5, rng_key=model_key)
# Analyze results
k_samples = [s.k for s in samples]
k_counts = np.bincount(k_samples)
print(f"Component distribution: {k_counts}")
print(f"Most likely k: {np.argmax(k_counts)}")

Iteration 0, k=2, log_prob=0.00
Iteration 100, k=2, log_prob=13140.01
Iteration 200, k=2, log_prob=14504.61
Iteration 300, k=2, log_prob=14504.61
Iteration 400, k=2, log_prob=14515.38
Iteration 500, k=2, log_prob=14515.38
Iteration 600, k=2, log_prob=14515.38
Iteration 700, k=2, log_prob=14515.38
Iteration 800, k=2, log_prob=14565.55
Iteration 900, k=2, log_prob=14591.66
Iteration 1000, k=2, log_prob=14636.03
Component distribution: [  0   0 549]
Most likely k: 2


In [13]:
# Load your data
rng_key=jax.random.PRNGKey(43)
rng_key, model_key, data_key = jax.random.split(rng_key, 3)

q_vals, t_vals, _ = prep_data_g2("~/repos/DLS/Experimental_data_083122/mix_1.csv")

true_params = {
    'beta': 1.0,
    'amp': jnp.array([0.5, 0.5]),
    'mu': jnp.array([1.3, 0.4]),
    'sig': jnp.array([0.25, 0.1]),
    'loss': 0.01,
    'noise_std': 0.001
}

true_g2 = g2_minus1_matrix(
    q_vals, t_vals,
    true_params['amp'], true_params['mu'], true_params['sig'], true_params['beta']
)
observed_g2 = true_g2 + true_params['noise_std'] * jax.random.normal(data_key, shape=true_g2.shape)

# Run RJMCMC
samples = run_rjmcmc(q_vals, t_vals, observed_g2, num_iterations=1100, max_k=5, rng_key=model_key)
# Analyze results
k_samples = [s.k for s in samples]
k_counts = np.bincount(k_samples)
print(f"Component distribution: {k_counts}")
print(f"Most likely k: {np.argmax(k_counts)}")

Iteration 0, k=2, log_prob=-4135199.92
Iteration 100, k=5, log_prob=-1520.79
Iteration 200, k=5, log_prob=7600.74
Iteration 300, k=5, log_prob=10766.23
Iteration 400, k=5, log_prob=11478.64
Iteration 500, k=5, log_prob=13564.95
Iteration 600, k=5, log_prob=13755.20
Iteration 700, k=5, log_prob=13755.20
Iteration 800, k=5, log_prob=13755.20
Iteration 900, k=5, log_prob=13788.02
Iteration 1000, k=5, log_prob=13788.02
Component distribution: [  0   0   0   0   0 549]
Most likely k: 5


In [12]:
samples[0].amp

Array([0.71678506, 0.28321494, 0.        , 0.        , 0.        ],      dtype=float64)